# [SK 05 - AssistantAPI AGENT with AzureAssistantAgent (plugin and CodeInterpreter)](https://learn.microsoft.com/en-us/semantic-kernel/frameworks/agent/agent-types/assistant-agent?pivots=programming-language-python)
## e.g. from `ChatCompletionAgent` to `AzureAssistantAgent` (or `OpenAIAssistantAgent`)
Now we learn how to use `Assistants API's` with `Semantic Kernel`.<br/>
Key difference betweeen Chat Completion API and Assistants API:
- Chat Completion API focuses on simple conversational exchanges
- Assistant API allows for dynamic, goal-driven interactions with additional features like code-interpreter and file-search

[Samples](https://github.com/microsoft/semantic-kernel/tree/main/python/samples/getting_started_with_agents)

# [Migration guide](https://learn.microsoft.com/en-us/semantic-kernel/support/migration/agent-framework-rc-migration-guide?pivots=programming-language-python)
### **Semantic Kernel** `ChatCompletionAgent` vs **(Azure) OpenAI** `AzureAssistantAgent` / `OpenAIAssistantAgent`
Here we show how to create an **OpenAI Assistant** using Azure OpenAI, but may do the same thing with the basic OpenAI service.<br/>
Regardless of the framework used (Semantic Kernel in this case), the `AzureAssistantAgent` as well as `OpenAIAssistantAgent` are **OpenAI Assistants**, which allow for **function calling**, the use of **file search** and a **code interpreter**.<br/>
Assistant **Threads** are used to manage the **conversation state**, similar to a Semantic Kernel Chat History **but hosted on the service side**, and automatically managed by the OpenAI service.

# Constants and Libraries

In [1]:
import os
from dotenv import load_dotenv # requires python-dotenv

load_dotenv("./../config/credentials_my.env")

agent_name                  = "mauromi_assistant_agent_python"
chatcompletion_service_id   = "chatcompletion_service_id"
instructions                = "you are a clever agent"
question_plugin             = "Toggle the status of my second light."
question_plugin_followup    = "Toggle the first light and give me the status of all my lights."
question_search             = "Using the free text file about product information, pls tell me how much for the Trail Master."
plugin_name                 = "Lights"
file_to_search_in           = "./data/product_info_1.md"

# deployment_name = os.environ['MODEL_DEPLOYMENT_NAME']

print(f"os.environ['AZURE_OPENAI_ENDPOINT']: {os.environ['AZURE_OPENAI_ENDPOINT']}")

os.environ['AZURE_OPENAI_ENDPOINT']: https://mmoaiswc-01.openai.azure.com/


# Set the logging level for  semantic_kernel.kernel to DEBUG
One of the main benefits of using Semantic Kernel is that it supports enterprise-grade services.<br/>
In this sample, we add the logging service to the kernel to help debug the AI agent.

In [2]:
import logging

logging.basicConfig(
    format="[%(asctime)s - %(name)s:%(lineno)d - %(levelname)s] %(message)s",
    datefmt="%Y-%m-%d %H:%M:%S")
    
logging.getLogger("kernel").setLevel(logging.DEBUG)
logging.getLogger().addHandler(logging.StreamHandler())
logging.getLogger().setLevel(logging.ERROR) # or logging.DEBUG to see more details

# 1. Setting up Resources: [`Assistant Client`](https://learn.microsoft.com/en-us/semantic-kernel/support/migration/agent-framework-rc-migration-guide?pivots=programming-language-python#setting-up-resources)

In [3]:
from semantic_kernel.agents import AzureAssistantAgent

# Set up the client and model using Azure OpenAI Resources
client = AzureAssistantAgent.create_client()

print(f"Base url: {client.base_url}")

Base url: https://mmoaiswc-01.openai.azure.com/openai/


# 2. Setting up Resources: `AzureOpenAISettings`
Now that we have the assistant client created, the call to AzureOpenAISettings returns the settings associated with the environment variables.<br/>
If we do it before creating the client, it does not capture all the proper settings.

In [4]:
from semantic_kernel.connectors.ai.open_ai import AzureOpenAISettings

ai_settings = AzureOpenAISettings()
ai_settings

AzureOpenAISettings(env_file_path=None, env_file_encoding='utf-8', chat_deployment_name='gpt-4o', responses_deployment_name=None, text_deployment_name='gpt-35-turbo-instruct', embedding_deployment_name='text-embedding-ada-002', text_to_image_deployment_name=None, audio_to_text_deployment_name=None, text_to_audio_deployment_name=None, realtime_deployment_name=None, endpoint=AnyUrl('https://mmoaiswc-01.openai.azure.com/'), base_url=None, api_key=SecretStr('**********'), api_version='2025-04-01-preview', token_endpoint='https://cognitiveservices.azure.com/.default')

# 3. Create CodeInterpreter tool and resources
Upload files into the assistant and retrieve code interpreter tool and resources.

In [5]:
# Upload the files to the client
file_ids: list[str] = []
for path in [file_to_search_in]:
    with open(path, "rb") as file:
        uploaded_file = await client.files.create(file=file, purpose="assistants")
        file_ids.append(uploaded_file.id)
        print(f"File {path} uploaded as {uploaded_file.id}\n")

# Get the code interpreter tool and resources
code_interpreter_tools, code_interpreter_tool_resources = AzureAssistantAgent.configure_code_interpreter_tool(file_ids = file_ids)

print(f"code_interpreter_tools: {code_interpreter_tools}\ncode_interpreter_tool_resources: {code_interpreter_tool_resources}")

File ./data/product_info_1.md uploaded as assistant-EwEnyBxcWNQwftpXgYGuHB

code_interpreter_tools: [{'type': 'code_interpreter'}]
code_interpreter_tool_resources: {'code_interpreter': {'file_ids': ['assistant-EwEnyBxcWNQwftpXgYGuHB']}}


# 4. Creating the Open AI Assistant, e.g. the `Agent definition` for SK Agent
Notes:
- This command actually creates the the **OpenAI** `Assistant` in the remote service.
- However, this is still **not** a `Semantic Kernel` Agent, but will be used as a `definition` to create it.
- This assistant still does **not** contain the `plugin` (we could do it, but we'll do it later).

In [6]:
from semantic_kernel.connectors.ai.open_ai import AzureOpenAISettings

# the openai_assistant is normally defined as agent "definition"
openai_assistant = await client.beta.assistants.create(
    model=ai_settings.chat_deployment_name,
    instructions=instructions,
    name=agent_name,
    tools=code_interpreter_tools,
    tool_resources=code_interpreter_tool_resources,
)

print(f"The following OpenAI Assistant has been created: {openai_assistant}")

The following OpenAI Assistant has been created: Assistant(id='asst_MAdWfICn2uHoIKYimMq2APY5', created_at=1754743386, description=None, instructions='you are a clever agent', metadata={}, model='gpt-4o', name='mauromi_assistant_agent_python', object='assistant', tools=[CodeInterpreterTool(type='code_interpreter')], response_format='auto', temperature=1.0, tool_resources=ToolResources(code_interpreter=ToolResourcesCodeInterpreter(file_ids=['assistant-EwEnyBxcWNQwftpXgYGuHB']), file_search=None), top_p=1.0)


# 5. Creating an SK [Assistant Agent](https://learn.microsoft.com/en-us/semantic-kernel/support/migration/agent-framework-rc-migration-guide?pivots=programming-language-python#1-creating-an-assistant) based on the OpenAI Assistant definition
As we can see, this agent already ontains the associations with CodeInterpreter and Resources (=files to search in) 

In [7]:
agent = AzureAssistantAgent(
    client=client,
    definition=openai_assistant,
)

agent

AzureAssistantAgent(arguments=None, description=None, id='asst_MAdWfICn2uHoIKYimMq2APY5', instructions='you are a clever agent', kernel=Kernel(retry_mechanism=PassThroughWithoutRetry(), services={}, ai_service_selector=<semantic_kernel.services.ai_service_selector.AIServiceSelector object at 0x0000029172CD2270>, plugins={}, function_invocation_filters=[], prompt_rendering_filters=[], auto_function_invocation_filters=[]), name='mauromi_assistant_agent_python', prompt_template=None, client=<openai.lib.azure.AsyncAzureOpenAI object at 0x0000029172ABAA50>, definition=Assistant(id='asst_MAdWfICn2uHoIKYimMq2APY5', created_at=1754743386, description=None, instructions='you are a clever agent', metadata={}, model='gpt-4o', name='mauromi_assistant_agent_python', object='assistant', tools=[CodeInterpreterTool(type='code_interpreter')], response_format='auto', temperature=1.0, tool_resources=ToolResources(code_interpreter=ToolResourcesCodeInterpreter(file_ids=['assistant-EwEnyBxcWNQwftpXgYGuHB'])

# 6. Define native plugin and planner
Here we do the following:
- 6.1 Plugin definition
- 6.2 Plugin association with the SK Agent created in the previous step
- 6.3 Planner settings definition and association with the SK agent for automatic plugin invokation

## 6.1 Plugin definition

In [8]:
class LightsPlugin:
    from typing import Annotated
    from semantic_kernel.functions import kernel_function
   
    def __init__(self):
        self.lights = [
        {"id": 0, "name": "Table Lamp", "is_on": False},
        {"id": 1, "name": "Porch light", "is_on": False},
        {"id": 2, "name": "Chandelier", "is_on": False},]
 
    @kernel_function(
        name="get_lights", # <<<=== DIFFERENT FROM THE FUNCTION NAME <get_state>, which will be ignored
        description="Gets a list of lights and their current state",
    )
    def get_state(
        self,
    ) -> Annotated[str, "the output is a string"]:
        """Gets a list of lights and their current state."""
        return self.lights
 
    @kernel_function(
        name="change_state",
        description="Changes the state of the light",
    )
    def change_state(
        self,
        id: int,
        is_on: bool,
    ) -> Annotated[str, "the output is a string"]:
        """Changes the state of the light."""
        for light in self.lights:
            if light["id"] == id:
                light["is_on"] = is_on
                return light
        return None

## 6.2 Plugin association with the SK Agent created in the previous step
Now, the SK agent does contain the plugin association.

In [9]:
agent.plugins.append(LightsPlugin())
agent

AzureAssistantAgent(arguments=None, description=None, id='asst_MAdWfICn2uHoIKYimMq2APY5', instructions='you are a clever agent', kernel=Kernel(retry_mechanism=PassThroughWithoutRetry(), services={}, ai_service_selector=<semantic_kernel.services.ai_service_selector.AIServiceSelector object at 0x0000029172CD2270>, plugins={}, function_invocation_filters=[], prompt_rendering_filters=[], auto_function_invocation_filters=[]), name='mauromi_assistant_agent_python', prompt_template=None, client=<openai.lib.azure.AsyncAzureOpenAI object at 0x0000029172ABAA50>, definition=Assistant(id='asst_MAdWfICn2uHoIKYimMq2APY5', created_at=1754743386, description=None, instructions='you are a clever agent', metadata={}, model='gpt-4o', name='mauromi_assistant_agent_python', object='assistant', tools=[CodeInterpreterTool(type='code_interpreter')], response_format='auto', temperature=1.0, tool_resources=ToolResources(code_interpreter=ToolResourcesCodeInterpreter(file_ids=['assistant-EwEnyBxcWNQwftpXgYGuHB'])

## 6.3 Planner settings definition and association with the SK agent for automatic plugin invokation

In [10]:
from semantic_kernel.connectors.ai.open_ai.prompt_execution_settings.azure_chat_prompt_execution_settings import AzureChatPromptExecutionSettings
from semantic_kernel.connectors.ai.function_choice_behavior import FunctionChoiceBehavior # Auto(), Required() or NoneInvoke()
from semantic_kernel.functions.kernel_arguments import KernelArguments

execution_settings = AzureChatPromptExecutionSettings()
execution_settings.function_choice_behavior= FunctionChoiceBehavior.Auto() # Auto(), Required() or NoneInvoke()

agent.arguments=KernelArguments(settings=execution_settings)
execution_settings

AzureChatPromptExecutionSettings(service_id=None, extension_data={}, function_choice_behavior=FunctionChoiceBehavior(enable_kernel_functions=True, maximum_auto_invoke_attempts=5, filters=None, type_=<FunctionChoiceType.AUTO: 'auto'>), ai_model_id=None, frequency_penalty=None, logit_bias=None, max_tokens=None, number_of_responses=None, presence_penalty=None, seed=None, stop=None, stream=False, temperature=None, top_p=None, user=None, store=None, metadata=None, response_format=None, function_call=None, functions=None, messages=None, parallel_tool_calls=None, tools=None, tool_choice=None, structured_json_response=False, stream_options=None, max_completion_tokens=None, reasoning_effort=None, extra_body=None)

# 4. Create the Assistant Agent on Azure (AzureAssistantAgent)
**Possible bug**: Make sure that the constant value for DEFAULT_AZURE_API_VERSION in `envs\<semantic_kernel_env_name>\Lib\site-packages\semantic_kernel\connectors\ai\open_ai\const.py` matches your `OPENAI_API_VERSION` environment value (listed for example in credentials_my.env).

# Get ready to print all messages of a [`AssistantAgentThread`](https://learn.microsoft.com/en-us/semantic-kernel/frameworks/agent/examples/example-chat-agent?pivots=programming-language-python) object
The history is managed through a `thread` object.
<br/><br/>
The `async for` syntax is specifically designed to iterate over asynchronous generators. It handles the yielding of items properly and allows you to collect them into a list or process them one by one.<br/>
By contrast, `await` is used for simple coroutines that return a single result, not for iterating through async generators.

In [11]:
from semantic_kernel.agents import AssistantAgentThread

async def print_messages(thread: AssistantAgentThread):
    i=0
    messages = list(reversed([message async for message in thread.get_messages()])) # this doesn't work: messages = await thread.get_messages()
    for cmc in messages: # ChatMessageContent
        if cmc.inner_content is None:
            i += 1
            if cmc.role.value == "user" or cmc.role.value == "assistant":
                print(f"{i} - Role: {cmc.role.value}, text: {cmc.items[0].text}")
            elif cmc.role.value == "tool":
                print(f"{i} - Role: {cmc.role.value}, function_name: {cmc.items[0].name}")
        else:
            for choice in cmc.inner_content.choices:
                if choice.message.tool_calls is None:
                    i += 1
                    print(f"{i} - Finish reason: {choice.finish_reason}")
                else:
                    for tc in choice.message.tool_calls:
                        i += 1
                        print (f"{i} - Function call: {tc.function.name}({tc.function.arguments})")
    return 

# [Invoke the Agent using a thread](https://learn.microsoft.com/en-us/semantic-kernel/support/migration/agent-framework-rc-migration-guide?pivots=programming-language-python#2-creating-a-thread)
As of Semantic Kernel Python 1.26.0 and later, we introduced a new common abstraction to manage threads for all agents. For each agent we now expose a thread class that implements the `AgentThread` base class, allowing context management via methods like create() and delete().

In [12]:
from semantic_kernel.agents import AssistantAgentThread

thread = AssistantAgentThread(client=client)

responses = agent.invoke(messages=question_plugin, thread=thread)

i=0
async for response in responses:
    i += 1
    print(f"\n++++++++++++++ RESPONSE {i} ++++++++++++++")
    print(response)
    thread = response.thread

print("\n\n++++++++++++++ MESSAGES ++++++++++++++")
await print_messages(thread=thread)


++++++++++++++ RESPONSE 1 ++++++++++++++
The status of your second light, "Porch light," has been toggled and is now on.


++++++++++++++ MESSAGES ++++++++++++++
1 - Role: user, text: Toggle the status of my second light.
2 - Role: assistant, text: The status of your second light, "Porch light," has been toggled and is now on.


In [13]:
responses = agent.invoke(messages=question_plugin, thread=thread)

i=0
async for response in responses:
    i += 1
    print(f"\n++++++++++++++ RESPONSE {i} ++++++++++++++")
    print(response)
    thread = response.thread

print("\n\n++++++++++++++ MESSAGES ++++++++++++++")
await print_messages(thread=thread)


++++++++++++++ RESPONSE 1 ++++++++++++++
The status of your second light, "Porch light," has been toggled and is now off.


++++++++++++++ MESSAGES ++++++++++++++
1 - Role: user, text: Toggle the status of my second light.
2 - Role: assistant, text: The status of your second light, "Porch light," has been toggled and is now on.
3 - Role: user, text: Toggle the status of my second light.
4 - Role: assistant, text: The status of your second light, "Porch light," has been toggled and is now off.


In [14]:
from semantic_kernel.agents import AssistantAgentThread

thread = AssistantAgentThread(client=client)
    
responses = agent.invoke(messages=question_search, thread=thread)

i=0
async for response in responses:
    i += 1
    print(f"\n++++++++++++++ RESPONSE {i} ++++++++++++++")
    print(response)
    thread = response.thread

print("\n\n++++++++++++++ MESSAGES ++++++++++++++")
await print_messages(thread=thread)


++++++++++++++ RESPONSE 1 ++++++++++++++
# Let's open the uploaded file and read its contents to find information about the Trail Master product.
file_path = '/mnt/data/assistant-EwEnyBxcWNQwftpXgYGuHB'

with open(file_path, 'r', encoding='utf-8') as file:
    file_contents = file.read()

file_contents

++++++++++++++ RESPONSE 2 ++++++++++++++
The Trail Master X4 Tent is priced at $250.


++++++++++++++ MESSAGES ++++++++++++++
1 - Role: user, text: Using the free text file about product information, pls tell me how much for the Trail Master.
2 - Role: assistant, text: The Trail Master X4 Tent is priced at $250.


# TEARDOWN

In [15]:
# just for testing: upload a file

file_path = "./skprompt.txt"

# Load the file as a FileObject
with open(file_path, "rb") as file:
    file = await client.files.create(file=file, purpose="assistants")

In [16]:
# list files

files_list = await client.files.list()
[(f.id, f.filename) for f in files_list.data]

[('assistant-YP4FBfBotkeFjQTHFBLu8w', 'skprompt.txt'),
 ('assistant-EwEnyBxcWNQwftpXgYGuHB', 'product_info_1.md'),
 ('assistant-3DCUaEQ7MeJHpsjtaQxGhU', 'skprompt.txt')]

In [17]:
# delete all files
async def print_files(client):
    files_list = await client.files.list()  # Fetch the most up-to-date list
    i = 0
    for file in files_list.data:
        i += 1
        print(f"File {i}: {file.filename} (id={file.id}) is being deleted...")
        await client.files.delete(file_id=file.id)
    print(f"{i} files were deleted")

await print_files(client)

File 1: skprompt.txt (id=assistant-YP4FBfBotkeFjQTHFBLu8w) is being deleted...
File 2: product_info_1.md (id=assistant-EwEnyBxcWNQwftpXgYGuHB) is being deleted...
File 3: skprompt.txt (id=assistant-3DCUaEQ7MeJHpsjtaQxGhU) is being deleted...
3 files were deleted


In [18]:
# delete thread

print(f"Deleting thread {thread.id}...")
await thread.delete()

Deleting thread thread_JX94fi7bnfvkgXa2aTKRN3u0...


In [19]:
# delete all assistant agents

assistants_list = (await client.beta.assistants.list()).data
print(f"There are {len(assistants_list)} assistant(s) to delete")

i = 0
for assistant in assistants_list:
    i += 1
    print(f"Assistant {i}/{len(assistants_list)}: Assistant {assistant.name} ({assistant.id})) is being deleted...")
    await client.beta.assistants.delete(assistant_id=assistant.id) # comment / un-comment this line if you want to delete it

There are 1 assistant(s) to delete
Assistant 1/1: Assistant mauromi_assistant_agent_python (asst_MAdWfICn2uHoIKYimMq2APY5)) is being deleted...
